# Heart Disease Prediction using Machine Learning and Deep Learning

**Student:** Aarambh Saxena  |  **Course:** BCA (AI Analyst)  |  **Guide:** Hritik Nayak

**Goal:** Predict whether a patient has heart disease (`HeartDisease` = 1) or not (0) from 11 clinical features.

**Dataset:** [Heart Failure Prediction (Kaggle, fedesoriano)](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction)

**Plan:**
1. Load and understand the data
2. Clean and explore (EDA)
3. Preprocess (impute, scale, one-hot encode)
4. Train and compare ML models with cross-validation
5. Train a Deep Learning model (Keras MLP)
6. Evaluate everything on the same held-out test set
7. Save the best model

## 1. Setup

In [ ]:
!pip install -q kagglehub

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import tensorflow as tf
from tensorflow import keras

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__)

## 2. Load the dataset from Kaggle

`kagglehub` downloads this public dataset directly in Colab. If it fails, download `heart.csv` from the Kaggle page,
upload it to Colab (Files panel) and set `CSV_PATH = "heart.csv"`.

In [ ]:
try:
    import kagglehub
    path = kagglehub.dataset_download("fedesoriano/heart-failure-prediction")
    CSV_PATH = os.path.join(path, "heart.csv")
except Exception as e:
    print("kagglehub failed:", e)
    CSV_PATH = "heart.csv"   # manual upload fallback

df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

## 3. Understand and clean the data

In [ ]:
df.info()
print("\nMissing values:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df.describe().T

In [ ]:
# Class balance
counts = df["HeartDisease"].value_counts()
print(counts)
print((counts / len(df) * 100).round(2).astype(str) + " %")

**Data-quality check.** In this dataset `Cholesterol = 0` and `RestingBP = 0` are not possible medical values;
they are placeholders for missing data. We convert them to `NaN` and impute them later **inside the pipeline**
(using training data only), which avoids data leakage.

In [ ]:
for col in ["Cholesterol", "RestingBP"]:
    print(col, "zeros:", (df[col] == 0).sum())
    df[col] = df[col].replace(0, np.nan)
df.isna().sum()

## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.countplot(data=df, x="HeartDisease", ax=axes[0]); axes[0].set_title("Target distribution")
sns.histplot(data=df, x="Age", hue="HeartDisease", kde=True, ax=axes[1]); axes[1].set_title("Age vs heart disease")
sns.boxplot(data=df, x="HeartDisease", y="MaxHR", ax=axes[2]); axes[2].set_title("Max heart rate vs heart disease")
plt.tight_layout(); plt.show()

In [ ]:
cat_cols_eda = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]
fig, axes = plt.subplots(1, len(cat_cols_eda), figsize=(20, 4))
for ax, c in zip(axes, cat_cols_eda):
    rate = df.groupby(c)["HeartDisease"].mean().mul(100)
    rate.plot(kind="bar", ax=ax)
    ax.set_title(f"Disease rate by {c}"); ax.set_ylabel("% with heart disease")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.select_dtypes("number").corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation of numeric features"); plt.show()

**After running the plots, note your observations:**
- Which age range shows more cases in this dataset?
- How does MaxHR differ between the two target classes?
- Which categories show higher disease rates in the bar charts?

These are associations in this dataset, not medical conclusions or causation.


## 5. Train / test split and preprocessing

In [ ]:
X = df.drop(columns="HeartDisease")
y = df["HeartDisease"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Disease rate  train: %.3f  test: %.3f" % (y_train.mean(), y_test.mean()))

num_cols = ["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak"]
bin_cols = ["FastingBS"]
cat_cols = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols),
    ("bin", "passthrough", bin_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])

## 6. Machine Learning models (5-fold cross-validation on training data)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=6,
                                            min_samples_leaf=3, random_state=SEED),
    "Gradient Boosting": GradientBoostingClassifier(random_state=SEED),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

cv_rows = []
for name, m in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", m)])
    s = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    cv_rows.append({"Model": name, **{k: s[f"test_{k}"].mean().round(3) for k in scoring}})

cv_table = pd.DataFrame(cv_rows).set_index("Model")
cv_table

The model with the best cross-validated F1 / ROC-AUC is chosen as the best ML model. The test set is **not** used for this choice.

In [ ]:
best_ml_name = cv_table["f1"].idxmax()
print("Best ML model by CV F1:", best_ml_name)

ml_pipe = Pipeline([("prep", clone(preprocessor)), ("model", clone(models[best_ml_name]))])
ml_pipe.fit(X_train, y_train)

## 7. Deep Learning model (Keras MLP)

In [ ]:
# Fit preprocessing on TRAIN only, then transform both sets
dl_prep = clone(preprocessor).fit(X_train)
Xtr = dl_prep.transform(X_train).astype("float32")
Xte = dl_prep.transform(X_test).astype("float32")
print("Input features after encoding:", Xtr.shape[1])

def build_mlp(n_features):
    model = keras.Sequential([
        keras.layers.Input(shape=(n_features,)),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy",
                  metrics=["accuracy", keras.metrics.AUC(name="auc")])
    return model

mlp = build_mlp(Xtr.shape[1])
mlp.summary()

In [ ]:
early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15,
                                      restore_best_weights=True)
history = mlp.fit(Xtr, y_train.values, validation_split=0.2,
                  epochs=200, batch_size=32, callbacks=[early], verbose=0)
print("Stopped after", len(history.history["loss"]), "epochs")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="train"); axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(history.history["accuracy"], label="train"); axes[1].plot(history.history["val_accuracy"], label="validation")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

## 8. Final evaluation on the held-out test set

In [ ]:
def metrics(y_true, y_pred, y_score):
    return {"Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred),
            "Recall": recall_score(y_true, y_pred),
            "F1": f1_score(y_true, y_pred),
            "ROC-AUC": roc_auc_score(y_true, y_score)}

ml_score = ml_pipe.predict_proba(X_test)[:, 1]
ml_pred = (ml_score >= 0.5).astype(int)

dl_score = mlp.predict(Xte, verbose=0).ravel()
dl_pred = (dl_score >= 0.5).astype(int)

results = pd.DataFrame({f"{best_ml_name} (ML)": metrics(y_test, ml_pred, ml_score),
                        "Keras MLP (DL)": metrics(y_test, dl_pred, dl_score)}).T.round(3)
results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, ml_pred), display_labels=["No disease", "Disease"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title(f"{best_ml_name}")
ConfusionMatrixDisplay(confusion_matrix(y_test, dl_pred), display_labels=["No disease", "Disease"]).plot(ax=axes[1], colorbar=False)
axes[1].set_title("Keras MLP")
plt.tight_layout(); plt.show()

print(classification_report(y_test, ml_pred, target_names=["No disease", "Disease"]))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, ml_score, name=best_ml_name, ax=ax)
RocCurveDisplay.from_predictions(y_test, dl_score, name="Keras MLP", ax=ax)
ax.plot([0, 1], [0, 1], "k--", alpha=0.4); ax.set_title("ROC curves (test set)")
plt.show()

**After running the evaluation, write a short conclusion using the actual test metrics:** compare the ML model and MLP, mention the stronger metric(s), and explain why recall and false negatives are important to consider in medical screening. Do not claim the model is suitable for diagnosis.


## 9. Which features matter? (tree-based models only)

In [ ]:
final_est = ml_pipe.named_steps["model"]
if hasattr(final_est, "feature_importances_"):
    names = ml_pipe.named_steps["prep"].get_feature_names_out()
    imp = pd.Series(final_est.feature_importances_, index=names).sort_values().tail(12)
    imp.plot(kind="barh", figsize=(7, 5), title="Top feature importances")
    plt.show()
else:
    print("Feature importances are shown only for tree models; best model here is", best_ml_name)

## 10. Save the model and predict for a new patient

In [ ]:
joblib.dump(ml_pipe, "heart_model.joblib")
mlp.save("heart_mlp.keras")

new_patient = pd.DataFrame([{
    "Age": 55, "Sex": "M", "ChestPainType": "ASY", "RestingBP": 140, "Cholesterol": 230,
    "FastingBS": 0, "RestingECG": "Normal", "MaxHR": 120, "ExerciseAngina": "Y",
    "Oldpeak": 1.5, "ST_Slope": "Flat"}])
p = ml_pipe.predict_proba(new_patient)[0, 1]
print(f"Predicted probability of heart disease: {p:.2f}  ->  {'Disease' if p >= 0.5 else 'No disease'}")

## 11. Limitations
- Small dataset (918 rows) combined from several sources; results may not transfer to other hospitals.
- Model scores are not calibrated clinical probabilities. This is a learning project, **not** a medical tool.
- No hyperparameter search was done; thresholds were left at 0.5.